In [ ]:
from google.colab import drive

import os
import json
import joblib
import numpy as np
import pandas as pd

drive.mount("/content/drive")

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

rf_model_10 = joblib.load(
    os.path.join(
        MODEL_PATH,
        "phase12_5_10_station_random_forest.joblib"
    )
)

FINAL_STATIONS = [
    "DL031",
    "DL002",
    "DL021",
    "DL028",
    "DL019",
    "DL010",
    "DL034",
    "DL007",
    "DL013",
    "DL004"
]

print("Random Forest loaded.")
print("Final stations:", len(FINAL_STATIONS))

In [ ]:
OGD_RESOURCE_ID = (
    "3b01bcb8-0b14-4abf-b6f2-"
    "c1bfd384ba69"
)

OGD_API_URL = (
    "https://api.data.gov.in/resource/"
    + OGD_RESOURCE_ID
)

print(OGD_API_URL)

In [ ]:
from getpass import getpass

OGD_API_KEY = getpass(
    "Enter your data.gov.in API key: "
)

if not OGD_API_KEY.strip():
    raise ValueError(
        "An OGD API key is required."
    )

print("API key received.")

In [ ]:
# ============================================================
# STEP 13.4 — FIXED OGD API CONNECTION TEST
# ============================================================

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Create a resilient HTTP session
session = requests.Session()

retry_strategy = Retry(
    total=4,
    connect=4,
    read=4,
    backoff_factor=2,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504
    ],
    allowed_methods=[
        "GET"
    ],
    raise_on_status=False
)

adapter = HTTPAdapter(
    max_retries=retry_strategy
)

session.mount(
    "https://",
    adapter
)

session.mount(
    "http://",
    adapter
)

# Small request first
request_params = {
    "api-key": OGD_API_KEY,
    "format": "json",
    "limit": 1
}

print(
    "Testing OGD endpoint..."
)

try:

    response = session.get(
        OGD_API_URL,
        params=request_params,

        # (connection timeout, read timeout)
        timeout=(
            15,
            90
        )
    )

    print(
        "HTTP status:",
        response.status_code
    )

    print(
        "Response size:",
        len(response.content),
        "bytes"
    )

    print(
        "Content type:",
        response.headers.get(
            "content-type",
            "unknown"
        )
    )

    if response.status_code != 200:

        print(
            "\nOGD server returned an error:"
        )

        print(
            response.text[:2000]
        )

    else:

        print(
            "\nOGD API connection successful."
        )

        print(
            response.text[:1000]
        )

except requests.exceptions.Timeout:

    print(
        "\nOGD API timed out again."
    )

    print(
        "The API server is currently too slow "
        "to respond from this Colab runtime."
    )

except requests.exceptions.RequestException as e:

    print(
        "\nOGD connection error:"
    )

    print(
        repr(e)
    )

In [ ]:
# ============================================================
# STEP 13.4B — OGD AVAILABILITY DIAGNOSTIC
# ============================================================

import requests
import time

diagnostic_urls = {
    "OGD_API_HOST":
        "https://api.data.gov.in/",

    "OGD_RESOURCE":
        OGD_API_URL,

    "OGD_WEB_RESOURCE":
        "https://www.data.gov.in/resource/"
        "real-time-air-quality-index-various-locations"
}

for name, url in diagnostic_urls.items():

    print("\n" + "=" * 60)
    print(name)
    print(url)

    start = time.time()

    try:

        r = session.get(
            url,
            timeout=(10, 30)
        )

        elapsed = time.time() - start

        print(
            "HTTP:",
            r.status_code
        )

        print(
            "Elapsed:",
            round(elapsed, 2),
            "seconds"
        )

        print(
            "Content length:",
            len(r.content)
        )

        if r.text:
            print(
                "Preview:",
                r.text[:300]
            )

    except Exception as e:

        elapsed = time.time() - start

        print(
            "ERROR after",
            round(elapsed, 2),
            "seconds"
        )

        print(
            repr(e)
        )

In [ ]:
# ============================================================
# PHASE 13 — SESSION CHECKPOINT
# ============================================================

import os
import json
import joblib

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

# ------------------------------------------------------------
# 1. Save API/source status
# ------------------------------------------------------------

phase13_checkpoint = {
    "completed_phase": "Phase 12.5 - Final 10-Station Model Comparison",

    "current_phase": "Phase 13 - CPCB/OGD Live Integration",

    "next_phase": (
        "Phase 13A - Live Adapter + Offline Replay"
    ),

    "production_candidate": "Random Forest",

    "station_count": 10,

    "forecast_target": "AQI_t_plus_1",

    "forecast_horizon_hours": 1,

    "final_random_forest": {
        "MAE": 3.497741,
        "RMSE": 8.457054,
        "R2": 0.995353
    },

    "ogd_resource_id": (
        "3b01bcb8-0b14-4abf-b6f2-"
        "c1bfd384ba69"
    ),

    "ogd_live_status": (
        "Temporarily unavailable / "
        "API gateway timeout and HTTP 500"
    ),

    "real_live_prediction_enabled": False,

    "parity_audit_passed": True,

    "next_action": (
        "Build live adapter and offline replay "
        "using historical 10-station observations."
    )
}

checkpoint_path = os.path.join(
    OUTPUT_PATH,
    "phase13_checkpoint.json"
)

with open(
    checkpoint_path,
    "w"
) as f:
    json.dump(
        phase13_checkpoint,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 2. Save session status
# ------------------------------------------------------------

session_status = {
    "completed_phase": "Phase 12.5",
    "current_phase": "Phase 13",
    "next_phase": "Phase 13A",
    "current_model": "Random Forest",
    "station_count": 10,
    "forecast": "AQI(t) -> AQI(t+1)",
    "ogd_available": False
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(
    status_path,
    "w"
) as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 3. Verify saved production model
# ------------------------------------------------------------

rf_path = os.path.join(
    MODEL_PATH,
    "phase12_5_10_station_random_forest.joblib"
)

print("=" * 65)
print("PHASE 13 CHECKPOINT SAVED")
print("=" * 65)

print(
    "Checkpoint:",
    checkpoint_path,
    "OK" if os.path.exists(checkpoint_path)
    else "ERROR"
)

print(
    "Session status:",
    status_path,
    "OK" if os.path.exists(status_path)
    else "ERROR"
)

print(
    "Production model:",
    rf_path,
    "OK" if os.path.exists(rf_path)
    else "ERROR"
)

print("\nCurrent model: Random Forest")
print("Stations: 10")
print("Target: AQI(t+1)")
print("OGD live connection: NOT AVAILABLE")
print("Next: Phase 13A - Live Adapter + Offline Replay")
print("=" * 65)